# Sephora Cloud Batch Inference

This notebook runs the saved Linear SVM + TF-IDF pipeline on a batch of Sephora reviews, compares its outputs with local reference predictions, and exports a CSV.

**Required files in the working directory (or `SEPHORA_WORKSPACE`):**
- `product_tfidf_LinearSVM.joblib`
- `sephora_batch_input_fixed.csv`
- `sephora_expected_predictions.csv`

**Environment:** Python 3.12.3; package versions in `requirements.txt`. The input CSV is read with GB18030 encoding and a known brand-name correction is applied to match the validated cloud workflow.

**Note:** The source files are not embedded in this notebook. The published notebook is a cleaned copy of the cloud workflow, not a new execution.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# ==========================================
# 1. Configuration
# ==========================================

WORKSPACE = os.environ.get("SEPHORA_WORKSPACE", os.getcwd())

MODEL_PATH = os.path.join(
    WORKSPACE, "product_tfidf_LinearSVM.joblib"
)

INPUT_PATH = os.path.join(
    WORKSPACE, "sephora_batch_input_fixed.csv"
)

EXPECTED_PATH = os.path.join(
    WORKSPACE, "sephora_expected_predictions.csv"
)

OUTPUT_PATH = os.path.join(
    WORKSPACE, "sephora_final_predictions.csv"
)

# ==========================================
# 2. Load trained model
# ==========================================

bundle = joblib.load(MODEL_PATH)

pipeline = bundle["pipeline"]
features = bundle["features"]
threshold = bundle["threshold"]

print("Model loaded successfully!")
print("Model: Linear SVM + TF-IDF + Product Features")
print("Threshold:", threshold)

# ==========================================
# 3. Load real Sephora reviews
# ==========================================

batch_reviews = pd.read_csv(
    INPUT_PATH,
    encoding="gb18030"
)

# Repair known brand encoding issues
batch_reviews["brand_name"] = (
    batch_reviews["brand_name"].replace({
        "Lanc?me": "Lancôme"
    })
)

# Validate input
missing_features = [
    col for col in features
    if col not in batch_reviews.columns
]

if missing_features:
    raise ValueError(
        f"Missing input features: {missing_features}"
    )

assert batch_reviews["experiment_row_id"].is_unique

print("\nInput loaded successfully!")
print("Total reviews:", len(batch_reviews))

# ==========================================
# 4. Run cloud batch inference
# ==========================================

X = batch_reviews[features]

scores = pipeline.decision_function(X)

labels = (scores >= threshold).astype(int)

results = pd.DataFrame({
    "experiment_row_id": batch_reviews["experiment_row_id"],
    "decision_score": scores,
    "predicted_label": labels
})

print("\nBatch inference completed!")
print("Predicted poor reviews:", int(labels.sum()))
print("Predicted non-poor reviews:", int((labels == 0).sum()))

display(results.head(10))

# ==========================================
# 5. Validate cloud vs local predictions
# ==========================================

expected = pd.read_csv(EXPECTED_PATH)

comparison = results.merge(
    expected,
    on="experiment_row_id",
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == len(results) == len(expected)

comparison["label_match"] = (
    comparison["predicted_label"]
    == comparison["predicted_poor"]
)

comparison["score_difference"] = (
    comparison["decision_score"] - comparison["score"]
).abs()

print("\n=== Cloud vs Local Validation ===")
print("Records compared:", len(comparison))
print(
    "Label agreement:",
    f"{comparison['label_match'].mean():.2%}"
)
print(
    "Maximum score difference:",
    comparison["score_difference"].max()
)
print(
    "Mean score difference:",
    comparison["score_difference"].mean()
)
print(
    "Differences > 0.01:",
    int((comparison["score_difference"] > 0.01).sum())
)

# ==========================================
# 6. Save final cloud predictions
# ==========================================

results.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nPredictions saved successfully!")
print("Output:", OUTPUT_PATH)
print("Completed at:", datetime.now(timezone.utc).isoformat())